In [2]:
import requests
from bs4 import BeautifulSoup
from typing import Dict, Any, List
import time
from tqdm import tqdm 
import json

In [3]:
import VOATibetan_utils

In [89]:
import requests
from bs4 import BeautifulSoup
from typing import Dict, Any, List, Tuple
import time




def convert_tibetan_date(tibetan_date: str) -> Tuple[int, int, int]:
    """
    Convert a Tibetan date string to year, month, day integers.
    
    Args:
        tibetan_date (str): Tibetan date string
        
    Returns:
        Tuple[int, int, int]: (year, month, day)
        
    Raises:
        ValueError: If date cannot be parsed properly
    """
    # Dictionary for Tibetan month names
    tibetan_months = {
        "དང་པོ": 1, "གཉིས་པ": 2, "གསུམ་པ": 3, "བཞི་པ": 4, "ལྔ་པ": 5, 
        "དྲུག་པ": 6, "བདུན་པ": 7, "བརྒྱད་པ": 8, "དགུ་པ": 9, "བཅུ་པ": 10,
        "བཅུ་གཅིག་པ": 11, "བཅུ་གཉིས་པ": 12
    }
    
    # Dictionary for Tibetan numerals
    tibetan_numerals = {
        "༠": "0", "༡": "1", "༢": "2", "༣": "3", "༤": "4",
        "༥": "5", "༦": "6", "༧": "7", "༨": "8", "༩": "9"
    }
    
    try:
        # Split the date components
        parts = tibetan_date.split("།")
        if len(parts) < 3:
            raise ValueError(f"Invalid date format: {tibetan_date}")
        
        # Extract month
        month_part = parts[0]
        month = None
        for month_name, month_num in tibetan_months.items():
            if month_name in month_part:
                month = month_num
                break
        
        # If month not found through dictionary lookup, try to handle specific cases
        if month is None:
            # Check for specific month patterns
            for i, (month_name, month_num) in enumerate(tibetan_months.items()):
                if month_name in month_part:
                    month = month_num
                    break
            
            # If still not found, try numeric patterns
            if month is None:
                for i in range(1, 13):
                    month_patterns = [
                        f"ཟླ་{i}", 
                        f"སྤྱི་ཟླ་{i}",
                        f"ཟླ་{tibetan_months.get(i, '')}"
                    ]
                    if any(pattern in month_part for pattern in month_patterns):
                        month = i
                        break
        
        # If still no month identified, raise error
        if month is None:
            raise ValueError(f"Could not identify month in: {month_part}")
        
        # Extract day
        day_text = parts[1].strip()
        day = ""
        for char in day_text:
            if char in tibetan_numerals:
                day += tibetan_numerals[char]
        
        if not day:
            raise ValueError(f"Could not extract day from: {day_text}")
        
        day = int(day)
        
        # Extract year
        year_text = parts[2].strip()
        year = ""
        for char in year_text:
            if char in tibetan_numerals:
                year += tibetan_numerals[char]
        
        if not year:
            raise ValueError(f"Could not extract year from: {year_text}")
        
        year = int(year)
        
        # Basic validation
        if not (1 <= month <= 12 and 1 <= day <= 31 and 1900 <= year <= 2100):
            raise ValueError(f"Date values out of reasonable range: {year}-{month}-{day}")
            
        return year, month, day
    
    except Exception as e:
        raise ValueError(f"Error parsing Tibetan date '{tibetan_date}': {str(e)}")




def extract_all_VOATibetan_article_links(url: str) -> Dict[str, Any]:
    """
    Extracts all article links from a given VOATibetan webpage.

    Args:
    url (str): The URL of the VOATibetan webpage containing article links.

    Returns:
    Dict[str, Any]: A dictionary containing article links and status details.
    """
    
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
    }
    final_response = {
        "Links": [],
        "Message": "Success",
        "Response": 200,
        "source_url": url
    }
    load_more = False
    
    try:
        start_time = time.time()
        response = requests.get(url, headers=headers, timeout=(5, 60-5))
        response.raise_for_status()
        end_time = time.time()
        if end_time - start_time > 50:
            print(f"This URL took more than 50s: {url}")
        soup = BeautifulSoup(response.content, 'html.parser')
        
        article_div = soup.find("div", class_="col-xs-12 col-md-8 col-lg-8 pull-left content-offset")
        if not article_div:
            raise ValueError("Could not find the main article container on the page.")
        first_link = article_div.find("div", class_="media-block")

        
        article_links = []
        link1 = check_media(first_link)
        article_links.append(link1)
        
        all_articles_div = article_div.find("ul", id="ordinaryItems")
        if not all_articles_div:
            raise ValueError("Could not find the each article container on the page.")
        
        all_articles = all_articles_div.find_all("div", class_="media-block")
        for article in all_articles:
            link = check_media(article)
            if link:
                article_links.append(link)

        final_response["Links"] = article_links

        load_more_span = soup.find("p", class_="buttons btn--load-more")
        if load_more_span:
            load_more = True
        else:
            get_last_date_span = article.find("span", class_="date date--mb date--size-3")
            if get_last_date_span:
                date_text_tib = get_last_date_span.text.strip() if get_last_date_span else ""
                year, month, day = convert_tibetan_date(date_text_tib)
                print(year, month, day)
                return final_response, load_more, [year, month, day]
            else:
                final_response, load_more, []

        return final_response, load_more, []
    
    except requests.Timeout:
        final_response["Message"] = "Request timed out"
        final_response["Response"] = 408
        return final_response, True, []
    except requests.RequestException as e:
        final_response["Message"] = f"An error occurred while fetching the webpage: {e}"
        final_response["Response"] = getattr(e.response, 'status_code', 500)
        return final_response, True, []
    except ValueError as e:
        final_response["Message"] = f"An error occurred while parsing the webpage: {e}"
        final_response["Response"] = getattr(e.response, 'status_code', 500)
        return final_response, True, []
    except Exception as e:
        final_response["Message"] = f"An unexpected error occurred: {e}"
        final_response["Response"] = 500
        return final_response, True, []



In [68]:
custom_url= "https://www.voatibetan.com/z/2253?p="
date_url = f"{date[0]}/{date[1]}/{date[2]}?p="
custom_url = custom_url.replace("?p=", "/")
custom_url = custom_url + date_url
custom_url

'https://www.voatibetan.com/z/2253/2023/6/12?p='

In [90]:
URL = "https://www.voatibetan.com/z/2253?p=100"
custom_url= "https://www.voatibetan.com/z/2253?p="

found_url_links, load_more, date = extract_all_VOATibetan_article_links(URL)
if load_more==False and len(found_url_links["Links"])!=0:
    if len(date) == 3:
        custom_url= "https://www.voatibetan.com/z/2253?p="
        date_url = f"{date[0]}/{date[1]}/{date[2]}?p="
        custom_url = custom_url.replace("?p=", "/")
        custom_url = custom_url + date_url
        print(custom_url)
    else:
        print("No new date data")


2023 6 12
https://www.voatibetan.com/z/2253/2023/6/12?p=


In [79]:
# found_url_links

In [25]:
load_more

False

In [58]:
date

[2023, 6, 12]

In [78]:
# URL = "https://www.voatibetan.com/z/223?p=10"
URL = "https://www.voatibetan.com/z/2253?p=100"

custom_url= "https://www.voatibetan.com/z/2253?p="

found_url_links, load_more, date = extract_all_VOATibetan_article_links(URL)
if load_more==False and len(found_url_links["Links"])!=0:
    if len(date) == 3:
        custom_url= "https://www.voatibetan.com/z/2253?p="
        date_url = f"{date[0]}/{date[1]}/{date[2]}?p="
        custom_url = custom_url.replace("?p=", "/")
        custom_url = custom_url + date_url
        print(custom_url)
    else:
        print("No new date data")
        return 
elif len(found_url_links["Links"])!=0:
    print("next")
else:
    print("page error")

2023 6 12
https://www.voatibetan.com/z/2253/2023/6/12?p=
next


In [ ]:
# def loop_article_page(total_page, custom_url, key_code):
#     """
    
#     """
#     return_file = {
#         "Data": [],
#         "message": "success",
#         "response": 200
#     }
#     All_url_links = {}
    
#     try:
#         for i in tqdm(range(0, total_page)):
#             final_url = custom_url + str(i) 
#             found_url_links, load_more, date = VOATibetan_utils.extract_all_VOATibetan_article_links(final_url)
#             key = key_code + str(i)
#             All_url_links[key] = found_url_links
#             found_url_links, load_more, date = extract_all_VOATibetan_article_links(URL)
#             if load_more==False and len(found_url_links["Links"])!=0:
#                 if len(date) == 3:
#                     custom_url= "https://www.voatibetan.com/z/2253?p="
#                     date_url = f"{date[0]}/{date[1]}/{date[2]}?p="
#                     custom_url = custom_url.replace("?p=", "/")
#                     custom_url = custom_url + date_url
#                     print(custom_url)
#                 else:
#                     print("No new date data")
                    
#             if load_more==False:
#                 print(f"Final page number: {i}")
#                 break
#             elif len(found_url_links["Links"])!=0:
#                 print("next")
#             else:
#                 print("page error")



                
#         return_file["Data"] = All_url_links
#         # print(final_url)
#         return return_file
    
#     except Exception as e:
#         return_file["Data"] = All_url_links
#         return_file["message"] = e
#         return_file["response"] = 404
#         return return_file

In [91]:
def loop_article_page(total_page, custom_url, key_code):
    """
    Loops through article pages on VOA Tibetan website and extracts article links.
    
    Args:
        total_page (int): Maximum number of pages to scrape, used as a safeguard
        custom_url (str): Base URL for the VOA Tibetan articles
        key_code (str): Prefix for the keys in the output dictionary
        
    Returns:
        dict: Dictionary containing scraped data, success message, and response code
    """
    
    return_file = {
        "Data": [],
        "message": "success",
        "response": 200
    }
    All_url_links = {}
    pbar = tqdm(total=total_page)

    try:
        page_index = 99  # Start from page 1 instead of 98
        key_index = 0
        pages_processed = 0
        
        while pages_processed < total_page:  # Use total_page parameter as a limit
            pbar.update(1)
            final_url = custom_url + str(page_index)
            try: 
                found_url_links, load_more, date = extract_all_VOATibetan_article_links(final_url)
                
                # Store the found links
                key = key_code + str(page_index) + "_" + str(key_index)
                All_url_links[key] = found_url_links
                
                # Check if we need to load more pages or change the date
                if not load_more:
                    if found_url_links["Links"] and len(date) == 3:
                        # Update URL to next date range
                        custom_url = "https://www.voatibetan.com/z/2253"
                        date_url = f"/{date[0]}/{date[1]}/{date[2]}?p="
                        custom_url = custom_url + date_url
                        print(f"Moving to new date range: {custom_url}")
                        page_index = 99  # Reset page index for new date
                    else:
                        # No more pages to load and no new date
                        print(f"Final page number: {page_index}")
                        print(f"Total pages extracted: {key_index}")
                        if len(date) == 3:
                            print(f"Last date: {date[0]}/{date[1]}/{date[2]} and URL: {custom_url}")
                        else:
                            print("No date information available")
                        break
                
                page_index += 1
                key_index += 1
                pages_processed += 1
                
            except Exception as e:
                print(f"Error in extract_all_VOATibetan_article_links() on page {page_index}: {e}")
                page_index += 1  # Move to next page even after error
                pages_processed += 1
                continue
                
        return_file["Data"] = All_url_links

        pbar.close()
        return return_file
            
    except Exception as e:
        return_file["Data"] = All_url_links
        return_file["message"] = str(e)  # Convert exception to string
        return_file["response"] = 404
        pbar.close()
        return return_file

In [92]:
total_page = 2000 # for custom check
custom_url= "https://www.voatibetan.com/z/2253?p="
article_tag = "བོད།"
key_code = "Page " + article_tag + " "
print(f"Page code: {key_code}")

all_links = loop_article_page(total_page, custom_url, key_code)


Page code: Page བོད། 



  0%|          | 3/2000 [00:10<1:57:12,  3.52s/it]

2023 6 12
Moving to new date range: https://www.voatibetan.com/z/2253/2023/6/12?p=


  0%|          | 7/2000 [15:57<75:42:11, 136.74s/it]

  0%|          | 4/2000 [00:15<2:24:14,  4.34s/it]

2021 9 1
Moving to new date range: https://www.voatibetan.com/z/2253/2021/9/1?p=


...


  0%|          | 8/2000 [00:37<2:52:29,  5.20s/it]

2013 2 26
Moving to new date range: https://www.voatibetan.com/z/2253/2013/2/26?p=



  0%|          | 9/2000 [00:41<2:47:36,  5.05s/it]

2011 10 3
Moving to new date range: https://www.voatibetan.com/z/2253/2011/10/3?p=


KeyboardInterrupt: 

In [17]:
def scrape_VOATibetan_article_content(url, Tags=""):
    """
    
    
    """


    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
    }
    
    final_response = {
        "data": {
            'title': "",
            'body': {"Audio": "", "Text": []},
            'meta_data': {'URL': url, 'Author': "", 'Date': "", 'Tags': [Tags]}
        },
        "Message": "Success",
        "Response": 200
    }
    
    try:
        # Make the request to the URL
        response = requests.get(url, headers=headers)
        response.raise_for_status()
        
        # Parse the page content with BeautifulSoup
        soup = BeautifulSoup(response.content, 'html.parser')
        
        # getting title, date, tags
        # section role="header"
        title_data = soup.find("div", class_="col-title col-xs-12 col-md-10 pull-right")        
        # Extract title
        if title_data:
            final_response['data']["title"] = title_data.get_text(strip=True)  
        else:
            title_data = soup.find("div", class_="col-title col-xs-12 col-lg-10 pull-right")
            if title_data:
                final_response['data']["title"] = title_data.get_text(strip=True)  
            else:
                final_response['data']["title"] = ""
            

        # publishing-details and date
        date_author = soup.find("div", class_="publishing-details")
        if date_author:
            author_div = date_author.find("span", class_="date")
            author = author_div.get_text(strip=True) if author_div else ""

            date_div = date_author.find("a", class_="links__item-link")
            date_time = date_div.get_text(strip=True) if date_div else ""
        else:
            author_div = ""
            date_time = None

        ### due to mistake we can see that autor and date variable was switched
        final_response['data']['meta_data']["Author"] =  date_time
        final_response['data']['meta_data']["Date"] = author

        
        # Extract audio content
        try:
            # Find the main audio link  wsw__embed
            audio_div = soup.find_all('div', class_='c-mmp__player')
            if len(audio_div):
                audio_sources = []
                # print(audio_div)
                for audio_div in audio_div:
                    # Find the audio element within each div
                    audio = audio_div.find('audio')
                    if audio and audio.get("src"):
                        audio_sources.append(audio.get("src"))
                final_response['data']['body']["Audio"] = audio_sources
            else:
                final_response['data']['body']["Audio"] = ""
        except AttributeError as e:
            final_response['data']['body']["Audio"] = ""
        
        # Extract body content
        try:
            # Find the main content div
            content_div = soup.find('div', class_='wsw')
            if content_div:
                # excluding wsw__embed elements which has all audio
                for embed in content_div.find_all(class_='wsw__embed'):
                    embed.decompose()
                # Extract all text content, 
                main_content = []
                # Get remaining text, excluding empty lines
                remaining_text = content_div.get_text(strip=True, separator='\n').split('\n')
                main_content.extend([text for text in remaining_text if text])
                
                final_response['data']['body']["Text"] = main_content
            else:
                final_response['data']['body']["Text"] = []
        except AttributeError as e:
            final_response['data']['body']["Text"] = []


        if final_response['data']['body']["Text"] == []:
            # print("Empty")
            content_div = soup.find('div', class_='intro m-t-md')
            if content_div:
                # Extract all text content, 
                main_content = []
                # Get remaining text, excluding empty lines
                remaining_text = content_div.get_text(strip=True, separator='\n').split('\n')
                main_content.extend([text for text in remaining_text if text])
                
                final_response['data']['body']["Text"] = main_content
            else:
                final_response['data']['body']["Text"] = []
            

        
        return final_response
    except requests.Timeout:
        final_response["Message"] = "Request timed out"
        final_response["Response"] = 408  # Request Timeout
        return final_response
        
    except requests.RequestException as e:
        final_response["Message"] = f"An error occurred while fetching the article: {str(e)}"
        final_response["Response"] = getattr(e.response, 'status_code', 500)
        return final_response
    
    except Exception as e:
        final_response["Message"] = f"An error occurred in code: {str(e)}"
        final_response["Response"] = 404
        return final_response


In [18]:
scrape_VOATibetan_article_content("https://www.voatibetan.com/a/international-scientists-found-that-himalayan-glaciers-could-have-lost-80-of-their-volume-by-2100/7146774.html")

{'data': {'title': 'ཚན་རིག་པ་ཚོས་ཧི་མ་ལ་ཡའི་འཁྱག་རོམ་བརྒྱ་ཆ་༨༠་ཞུ་རྒྱུའི་བློ་འཚབ།',
  'body': {'Audio': [],
   'Text': ['༄༅།། ཧི་མ་ལ་ཡའི་གངས་རི་དང་འཁྱག་རོམ་ནི མི་གྲངས་དུང་ཕྱུར་མང་པོའི་འཐུང་ཆུ་དང་ཞིང་ཆུ་འདྲེན་གཏོང་། ཨེ་ཤི་ཡའི་ཆུ་བོ་ཆེན་པོ་༡༢་ཀྱི་འབྱུང་ཁུངས་སུ་གྱུར། ཕྱི་ཟླ་༦་པའི་ཚེས་༢༠་ཉིན་རོ་ཡེ་ཊར་སི་གསར་འགྱུར་ལས་ཁང་གིས་གནས་ཚུལ་སྤེལ་བ་ལྟར་ན། རྒྱལ་སྤྱིའི་ཚན་རིག་པའི་རུ་ཁག་ཅིག་གིས་འཛམ་གླིང་གི་སའི་གོ་ལའི་ཚ་དྲོད་རྒྱས་ཏེ་ཧི་མ་ལ་ཡའི་གངས་རི་དང་འཁྱག་རོམ་རྣམས་ཞུ། ཆུ་ལོག་བརྒྱུགས་ཏེ་མི་གྲངས་དུང་ཕྱུར་༢་དང་བྱེ་བ་༤་ལྷག་ལ་འཐུང་ཆུ་ཆད་རྒྱུའི་ཉེན་ཁ་ཆེ། ༢༠༡༠་ནས་ད་ལྟའི་བར་དུ་ཧི་མ་ལ་ཡའི་འཁྱག་རོམ་གྱི་ཞུ་ཚད་བརྒྱ་ཆ་༦༥་ཇེ་མགྱོགས་སུ་ཕྱིན་ཏེ་ལོ་བཅུ་ཕྲག་མང་པོའི་ཟིན་ཐོ་བརྒལ་འདུག་ཅེས་སྙན་ཐོ་བཏོན། བལ་ཡུལ་གྱི་རྒྱལ་ས་ཀ་ཐ་མན་རྡུ་གྲོང་ཁྱེར་དུ་ཡོད་པའི་རྒྱལ་སྤྱིའི་མཉམ་འབྲེལ་རི་བོ་བདག་སྐྱོང་ལྟེ་གནས་ཁང་གི་ཁོར་ཡུག་ཚན་རིག་པ་ཧྥི་ལི་པ་སི་ཝེ་སི་ཊར་གྱིས་འཁྱག་རོམ་རྣམས་ཞུ་ཡི་འདུག འཁྱག་རོམ་དེ་རྣམས་ལོ་བརྒྱའི་ནང་དུ་ཞུ་ཚར་ས་རེད་ཅེས་སྐྱེ་ཁམས་ལ་དོ་ཕོག་ཚབས་ཆེན་ཞིག་འཕོག་བཞིན་པ་གསལ་པོར་བསྟན།།',
    'ཧི་མ་ལ་ཡའི་གངས་རི་ནི་བོད་དང་འབྲུག་ཡུལ། ཨབ་ག

In [19]:
URL = "https://www.voatibetan.com/a/house-of-representatives-and-senate-honour-sikyong-penpa-tsering-in-the-australian-parliament-/7145552.html"
scrape_VOATibetan_article_content(URL)

Empty


{'data': {'title': 'ཨོས་ཀྲེ་ལི་ཡའི་གྲོས་ཚོགས་གོང་འོག་གིས་སྲིད་སྐྱོང་སྤེན་ཚེ་རིང་ལ་གུས་བསུ་ཞུས་པ།',
  'body': {'Audio': ['https://voa-audio.voanews.eu/vti/2023/06/20/01000000-0aff-0242-f945-08db71c9658c.mp3'],
   'Text': ['བོད་མིའི་སྒྲིག་འཛུགས་ཀྱི་སྲིད་སྐྱོང་སྤེན་པ་ཚེ་རིང་ལགས་ཀྱིས་ཨོས་ཀྲེ་ལི་ཡའི་གྲོས་ཚོགས་གོང་འོག་གཉིས་ཀྱི་འཐུས་མི་དང་མཇལ་མོལ་གནང་འདུག་ཅིང་། སྐབས་དེར་གྲོས་ཚོགས་གོང་འོག་གཉིས་ནང་བོད་དོན་གྲོས་གཞི་ངོ་སྤྲོད་བྱས་པ་མ་ཟད། སྲིད་སྐྱོང་སྤེན་པ་ཚེ་རིང་ལགས་སུ་གུས་བསུ་ཞུས་འདུག་པའི་སྐོར་གྱི་གནས་ཚུལ་གསན་རོགས་གནང་།']},
  'meta_data': {'URL': 'https://www.voatibetan.com/a/house-of-representatives-and-senate-honour-sikyong-penpa-tsering-in-the-australian-parliament-/7145552.html',
   'Author': 'རྣམ་རྒྱལ་ཤསྟྲི།',
   'Date': '༢༡།༠༦།༢༠༢༣',
   'Tags': ['']}},
 'Message': 'Success',
 'Response': 200}

In [20]:
URL = "https://www.voatibetan.com/a/sakya-trichen-in-lumbini-buddhist-conclave/7900042.html"
scrape_VOATibetan_article_content(URL)

Empty


{'data': {'title': 'ལུམ་བྷི་ནི་རུ་དཔལ་སྐྱ་༧གོང་མ་ཁྲི་ཆེན་རྡོ་རྗེ་འཆང་ལ་མདོ་སྨད་ལྷ་སྡེ་མི་སྡེ་ཡོངས་ཀྱིས་བརྟནབརྟན་བཞུགས།',
  'body': {'Audio': ['https://voa-audio-ns.akamaized.net/vti/2024/12/13/e726a36a-4d40-4583-aed1-a8cb6a509d67.mp3'],
   'Text': ['སྟོན་པ་ཐུགས་རྗེ་ཅན་སྐུ་འཁྲུངས་སའི་གནས་ལུམ་བྷི་ནི་རུ་དཔལ་སྐྱ་༧གོང་མ་ཁྲི་ཆེན་རྡོ་རྗེ་འཆང་ལ་མདོ་སྨད་ལྷ་སྡེ་མི་སྡེ་ཡོངས་ཀྱིས་གྱ་སྟོན་རྟེན་འབྲེལ་དང་བརྟན་བཞུགས་བསྟར་འབུལ་ཞུས་འདུག་པའི་སྐོར། བལ་ཡུལ་ནས་འདི་གའི་གསར་འགོད་པ་ཀུན་བཟང་བསྟན་འཛིན་ལགས་ཀྱིས་སྙན་སྒྲོན་ཞུ་གནང་གི་རེད།']},
  'meta_data': {'URL': 'https://www.voatibetan.com/a/sakya-trichen-in-lumbini-buddhist-conclave/7900042.html',
   'Author': '',
   'Date': '༡༣།༡༢།༢༠༢༤',
   'Tags': ['']}},
 'Message': 'Success',
 'Response': 200}

In [21]:
URL = "https://www.voatibetan.com/a/tibetan-scholar-nordrang-urgen-passed-away-at-91-in-lhasa/7871878.html"
scrape_VOATibetan_article_content(URL)

{'data': {'title': 'བོད་ཀྱི་རིག་གནས་ཐད་མཛད་རྗེས་ཆེ་བའི་མཁས་དབང་ནོར་བྲང་ཨོ་རྒྱན་མཆོག་སྐུ་གཤེགས་པ།',
  'body': {'Audio': ['https://voa-audio-ns.akamaized.net/vti/2024/11/21/33c3c2c8-707d-4686-8c66-a2c24d30528b.mp3'],
   'Text': ['བོད་ཀྱི་སྐད་ཡིག་དང་རིག་གནས་ཐད་མཛད་རྗེས་ཆེ་བའི་མཁས་དབང་ནོར་བྲང་ཨོ་རྒྱན་མཆོག་དགུང་གྲངས་༩༡ ཐོག ཕྱི་ཟླ་བཅུ་གཅིག་པའི་ཚེས་༡༩ བོད་ནང་གི་ཕྱི་དྲོའི་ཚོུད་བདུན་དང་སྐར་མ་བདུན་གྱི་ཐོག་བོད་ཀྱི་རྒྱལ་ས་ལྷ་སར་སྐུ་གཤེགས་པའི་ཡིད་སྐྱོའི་གནས་ཚུལ་ཐོན་འདུག བོད་ཕྱི་ནང་གི་སྤྱི་ཚོགས་དྲ་ལམ་ཁག་ཏུ་དམ་པ་ཁོང་གི་མཛད་རྗེས་ལ་རྗེས་དྲན་དང་འབྲེལ་མྱ་ངན་ཞུ་བཞིན་ཡོད་པ་རེད། བཞུགས་སྒར་རྡ་རམ་ས་ལའི་སྐུ་བཅར་རྣམ་རྒྱལ་གྲྭ་ཚང་གི་སློབ་སྤྱི་དང་རྩོམ་པ་པོ་རྒན་ཐུབ་བསྟན་ཡར་འཕེལ་ལགས་ནི་མཁས་དབང་དམ་པ་ཁོང་ལ་རྒྱུས་མང་ཡོད་མཁན་དང་ཁོང་དང་ཕ་ཡུལ་གཅིག་པ་ཡིན་པ་རེད། ཁོང་གིས་མཁས་དབང་ནོར་བྲང་ཨོ་རྒྱན་མཆོག་གི་སྐོར་ལ་ངོ་སྤྲོད་གནང་སོང་།',
    'ཁོང་གིས་འགྲེལ་བརྗོད་གནང་བ་ལྟར་ན་མཁས་དབང་ནོར་བྲང་ཨོ་རྒྱན་མཆོག་ཆུང་དུས་སུ་གཞིས་རྩེ་བཀྲ་ཤིས་ལྷུན་པོའི་དཀྱིལ་ཁང་གྲྭ་ཚང་དུ་བཞུགས་ནས་དཀའ་ཆེན་ངག་དབང་ལགས་དང་བཀའ་ཆེན་བློ་བཟང་བཟོད་པ་སོགས་མཁས་པ་མང་པོ་བསྟེ

In [24]:
scrape_VOATibetan_article_content("https://www.voatibetan.com/a/tibetan-political-leader-urges-australian-government-to-sanction-china-on-human-rights-abuse-in-tibet/7147226.html")

{'data': {'title': '',
  'body': {'Audio': ['https://voa-audio.voanews.eu/vti/2023/06/21/01000000-c0a8-0242-62e5-08db72956adf.mp3'],
   'Text': ['No Content in the article']},
  'meta_data': {'URL': 'https://www.voatibetan.com/a/tibetan-political-leader-urges-australian-government-to-sanction-china-on-human-rights-abuse-in-tibet/7147226.html',
   'Author': '༢༢།༠༦།༢༠༢༣',
   'Date': 'ཀུན་བཟང་སྒྲོལ་མ།',
   'Tags': ['']}},
 'Message': 'Success',
 'Response': 200}

In [23]:

URL = "https://www.voatibetan.com/a/sakya-trichen-in-lumbini-buddhist-conclave/7900042.html"
scrape_VOATibetan_article_content(URL)

Empty


{'data': {'title': 'ལུམ་བྷི་ནི་རུ་དཔལ་སྐྱ་༧གོང་མ་ཁྲི་ཆེན་རྡོ་རྗེ་འཆང་ལ་མདོ་སྨད་ལྷ་སྡེ་མི་སྡེ་ཡོངས་ཀྱིས་བརྟནབརྟན་བཞུགས།',
  'body': {'Audio': ['https://voa-audio-ns.akamaized.net/vti/2024/12/13/e726a36a-4d40-4583-aed1-a8cb6a509d67.mp3'],
   'Text': ['སྟོན་པ་ཐུགས་རྗེ་ཅན་སྐུ་འཁྲུངས་སའི་གནས་ལུམ་བྷི་ནི་རུ་དཔལ་སྐྱ་༧གོང་མ་ཁྲི་ཆེན་རྡོ་རྗེ་འཆང་ལ་མདོ་སྨད་ལྷ་སྡེ་མི་སྡེ་ཡོངས་ཀྱིས་གྱ་སྟོན་རྟེན་འབྲེལ་དང་བརྟན་བཞུགས་བསྟར་འབུལ་ཞུས་འདུག་པའི་སྐོར། བལ་ཡུལ་ནས་འདི་གའི་གསར་འགོད་པ་ཀུན་བཟང་བསྟན་འཛིན་ལགས་ཀྱིས་སྙན་སྒྲོན་ཞུ་གནང་གི་རེད།']},
  'meta_data': {'URL': 'https://www.voatibetan.com/a/sakya-trichen-in-lumbini-buddhist-conclave/7900042.html',
   'Author': '',
   'Date': '༡༣།༡༢།༢༠༢༤',
   'Tags': ['']}},
 'Message': 'Success',
 'Response': 200}